In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import mlflow
import mlflow.xgboost
import pickle
import json

PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

features_df = pd.read_parquet(PROCESSED_DIR / "features.parquet")
print(f"Loaded: {features_df.shape}")

Loaded: (32287, 33)


In [2]:
FEATURE_COLS = [
    "home_elo", "away_elo", "elo_diff",
    "home_goals_for_5", "home_goals_against_5",
    "home_win_rate_5", "home_points_5",
    "away_goals_for_5", "away_goals_against_5",
    "away_win_rate_5", "away_points_5",
    "goals_for_diff_5", "goals_against_diff_5",
    "win_rate_diff_5", "points_diff_5",
    "goals_for_diff_10", "win_rate_diff_10", "points_diff_10",
    "h2h_matches", "h2h_home_win_rate",
    "h2h_draw_rate", "h2h_away_win_rate",
    "is_friendly", "is_worldcup",
    "tournament_weight", "neutral",
]

le = LabelEncoder()
features_df["outcome_encoded"] = le.fit_transform(features_df["outcome"])

print("Label encoding:")
for i, cls in enumerate(le.classes_):
    print(f"  {i} = {cls}")

print(f"\nFeature columns: {len(FEATURE_COLS)}")

Label encoding:
  0 = away_win
  1 = draw
  2 = home_win

Feature columns: 26


In [3]:
# train on everything before WC 2022, test on WC 2022 onwards
train = features_df[features_df["date"] < "2022-11-20"].copy()
test  = features_df[features_df["date"] >= "2022-11-20"].copy()

X_train = train[FEATURE_COLS]
y_train = train["outcome_encoded"]

X_test = test[FEATURE_COLS]
y_test = test["outcome_encoded"]

print(f"Train: {len(train)} matches (up to Nov 2022)")
print(f"Test:  {len(test)} matches (WC 2022 + recent)")
print(f"\nTrain outcome split:")
print(train["outcome"].value_counts())
print(f"\nTest outcome split:")
print(test["outcome"].value_counts())

Train: 28582 matches (up to Nov 2022)
Test:  3705 matches (WC 2022 + recent)

Train outcome split:
outcome
home_win    13902
away_win     7941
draw         6739
Name: count, dtype: int64

Test outcome split:
outcome
home_win    1750
away_win    1104
draw         851
Name: count, dtype: int64


In [4]:
mlflow.set_experiment("worldcup-predictor")

with mlflow.start_run(run_name="xgb_v1"):

    params = {
        "n_estimators":     500,
        "max_depth":        4,
        "learning_rate":    0.05,
        "subsample":        0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 3,
        "gamma":            0.1,
        "reg_alpha":        0.1,
        "reg_lambda":       1.0,
        "objective":        "multi:softprob",
        "num_class":        3,
        "eval_metric":      "mlogloss",
        "random_state":     42,
    }

    model = xgb.XGBClassifier(**params)

    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=100,
    )

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    print(f"\nTest Accuracy: {acc:.4f} ({acc*100:.1f}%)")
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred,
                                target_names=le.classes_))

    mlflow.log_params(params)
    mlflow.log_metric("test_accuracy", acc)
    mlflow.xgboost.log_model(model, "model")

[0]	validation_0-mlogloss:1.04104
[100]	validation_0-mlogloss:0.87488
[200]	validation_0-mlogloss:0.87427
[300]	validation_0-mlogloss:0.87476
[400]	validation_0-mlogloss:0.87680
[499]	validation_0-mlogloss:0.87834


2026/06/24 19:37:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Test Accuracy: 0.5960 (59.6%)

Classification Report:
              precision    recall  f1-score   support

    away_win       0.57      0.62      0.59      1104
        draw       0.33      0.04      0.07       851
    home_win       0.62      0.85      0.72      1750

    accuracy                           0.60      3705
   macro avg       0.51      0.50      0.46      3705
weighted avg       0.54      0.60      0.53      3705



In [5]:
importance = pd.DataFrame({
    "feature":    FEATURE_COLS,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print("Feature importance ranking:")
print(importance.to_string(index=False))

Feature importance ranking:
             feature  importance
            elo_diff    0.218661
             neutral    0.072784
   h2h_home_win_rate    0.050886
      points_diff_10    0.047761
goals_against_diff_5    0.045594
         is_friendly    0.044993
   tournament_weight    0.044080
            away_elo    0.036155
   h2h_away_win_rate    0.033942
home_goals_against_5    0.033055
            home_elo    0.031411
away_goals_against_5    0.030626
       h2h_draw_rate    0.025599
   goals_for_diff_10    0.024930
         h2h_matches    0.023932
       away_points_5    0.023619
       home_points_5    0.023554
    away_goals_for_5    0.022973
    home_goals_for_5    0.022272
     away_win_rate_5    0.022088
    goals_for_diff_5    0.021734
       points_diff_5    0.021133
     win_rate_diff_5    0.020741
    win_rate_diff_10    0.020686
     home_win_rate_5    0.018915
         is_worldcup    0.017876


In [6]:
# save model
model_path = MODELS_DIR / "xgb_outcome_model.pkl"
with open(model_path, "wb") as f:
    pickle.dump(model, f)

# save label encoder
le_path = MODELS_DIR / "label_encoder.pkl"
with open(le_path, "wb") as f:
    pickle.dump(le, f)

# save feature columns
fc_path = MODELS_DIR / "feature_cols.json"
with open(fc_path, "w") as f:
    json.dump(FEATURE_COLS, f)

# save params for reference
params_path = MODELS_DIR / "model_params.json"
with open(params_path, "w") as f:
    json.dump({"version": "xgb_v1", "accuracy": acc, "params": params}, f, indent=2)

print(f"Saved model         → {model_path}")
print(f"Saved label encoder → {le_path}")
print(f"Saved feature cols  → {fc_path}")
print(f"Saved model params  → {params_path}")
print(f"\nModel version: xgb_v1")
print(f"Test accuracy: {acc*100:.1f}%")
print(f"\nNext: 08_predict.ipynb")

Saved model         → ..\models\xgb_outcome_model.pkl
Saved label encoder → ..\models\label_encoder.pkl
Saved feature cols  → ..\models\feature_cols.json
Saved model params  → ..\models\model_params.json

Model version: xgb_v1
Test accuracy: 59.6%

Next: 08_predict.ipynb
